In [ ]:
import pandas as pd
import json
from datetime import datetime
from typing import List, Dict, Any
import numpy as np
import os
import sys

raw_data_dir = "/content/data/raw/"
output_dir = "/content/data/fingerprints/"

# Funzione parse_scan_results
def parse_scan_results(data: Dict[str, Any]) -> pd.DataFrame:
    rows = []

    ble_data = data.get('bleData', {})
    for device_id, scans in ble_data.items():
        for scan in scans:
            scan_copy = scan.copy()
            scan_copy['device_id'] = device_id
            if 'serviceData' in scan_copy:
                
                try:
                    scan_copy['serviceData'] = json.dumps(scan_copy['serviceData'])
                except TypeError:
                    scan_copy['serviceData'] = str(scan_copy['serviceData'])
            rows.append(scan_copy)

    if not rows:
        return pd.DataFrame(columns=['rssi', 'timestamp', 'device_id'])

    df = pd.DataFrame(rows)
    if 'rssi' not in df.columns: df['rssi'] = np.nan
    if 'timestamp' not in df.columns: df['timestamp'] = pd.NaT
    if 'device_id' not in df.columns: df['device_id'] = None

    return df

# Funzioni create_radiomap e create_windows
def create_radiomap(df: pd.DataFrame, window_seconds: int = 3, window_step: int = 1, duration_seconds: int = None, default_rssi: float = 110.0) -> pd.DataFrame:
    if df.empty or 'timestamp' not in df.columns or df['timestamp'].isnull().all():
        return pd.DataFrame()

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').dropna(subset=['timestamp']) # Rimuovi righe senza timestamp
    df.replace(0, np.nan, inplace=True) # Sostituisci 0 con NaN solo dopo aver gestito i timestamp

    if df.empty: return pd.DataFrame()

    start_time = df['timestamp'].min()

    if duration_seconds is not None:
        num_windows = max(0, int((duration_seconds - window_seconds) / window_step) + 1)
    else:
        end_time = df['timestamp'].max()
        actual_duration = (end_time - start_time).total_seconds()
        num_windows = max(0, int((actual_duration - window_seconds) / window_step) + 1)

    windows = []
    all_device_ids = df['device_id'].unique()

    for window_idx in range(num_windows):
        window_start = start_time + pd.Timedelta(seconds=window_idx * window_step)
        window_end = window_start + pd.Timedelta(seconds=window_seconds)

        window_data = df[(df['timestamp'] >= window_start) & (df['timestamp'] < window_end)]

        mean_row = {'window_idx': window_idx}
        # Inizializza tutti i beacon a default_rssi
        for device_id in all_device_ids:
            mean_row[device_id] = default_rssi

        if not window_data.empty:
            # Calcola la MEDIANA invece della media
            # MODIFICA
            # Originale: rssi_means = window_data.groupby('device_id', as_index=False)['rssi'].mean()
            rssi_medians = window_data.groupby('device_id')['rssi'].median().reset_index()
            for _, row in rssi_medians.iterrows():
                 # Controlla se la mediana non è NaN prima di assegnare
                 if pd.notna(row['rssi']):
                     mean_row[row['device_id']] = row['rssi']
        # Altrimenti, lascia i valori di default_rssi

        windows.append(mean_row)

    radiomap = pd.DataFrame(windows)
    return radiomap

# Funzione create_windows
def create_windows(doc: Dict[str, Any], sensor: str, window_seconds: int = 3, window_step: int = 1, duration_seconds: int = None) -> pd.DataFrame:
    if sensor not in doc or not doc[sensor]:
        return pd.DataFrame()

    sensor_cols = ['x', 'y']

    df = pd.DataFrame(doc[sensor])
    if 'timestamp' not in df.columns or df['timestamp'].isnull().all():
        return pd.DataFrame()

    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.sort_values('timestamp').dropna(subset=['timestamp'])

    if df.empty: return pd.DataFrame()

    for col in sensor_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        else:
             df[col] = np.nan

    start_time = df['timestamp'].min()

    if duration_seconds is not None:
        num_windows = max(0, int((duration_seconds - window_seconds) / window_step) + 1)
    else:
        end_time = df['timestamp'].max()
        actual_duration = (end_time - start_time).total_seconds()
        num_windows = max(0, int((actual_duration - window_seconds) / window_step) + 1)

    windows = []

    for window_idx in range(num_windows):
        window_start = start_time + pd.Timedelta(seconds=window_idx * window_step)
        window_end = window_start + pd.Timedelta(seconds=window_seconds)

        window_data = df[(df['timestamp'] >= window_start) & (df['timestamp'] < window_end)]

        window_results = {
            'window_idx': window_idx,
        }

        default_sensor_values = {f'{sensor}_{col}_mean': float('nan') for col in sensor_cols}
        window_results.update(default_sensor_values)

        if not window_data.empty:
            for col in sensor_cols:
                data = window_data[col].dropna()
                if not data.empty: # Calcola la media solo se ci sono dati validi
                    window_results[f'{sensor}_{col}_mean'] = data.mean()
        # Altrimenti, lascia i NaN di default

        windows.append(window_results)

    sensor_windows = pd.DataFrame(windows)
    return sensor_windows


def process_json_data(docs: List[Dict[str, Any]], min_rssi_length: int = 100, window_seconds: int = 3, window_step: int = 1, sensors: List[str] = None, default_rssi: float = 110.0) -> pd.DataFrame:
    try:
        all_data = []
        all_beacon_columns = set() # Per tenere traccia di tutti i beacon visti

        processed_docs = []
        for doc in docs:
            artwork = doc.get("artwork")
            room = doc.get("room")
            platform = doc.get("platform", "unknown") # Default se manca
            duration = doc.get("recordingDurationSeconds")
            rssi_df = parse_scan_results(doc)

            if not rssi_df.empty and rssi_df.shape[0] > min_rssi_length:
                # Usa la versione modificata di create_radiomap che calcola la mediana
                radiomap = create_radiomap(rssi_df, window_seconds, window_step, duration, default_rssi)

                if not radiomap.empty:
                    # Aggiorna il set di tutti i beacon visti
                    current_beacons = set(col for col in radiomap.columns if col != 'window_idx')
                    all_beacon_columns.update(current_beacons)

                    sensor_dfs = {}
                    if sensors:
                        for sensor in sensors:
                            sensor_dfs[sensor] = create_windows(doc, sensor, window_seconds, window_step, duration)

                    combined_df = radiomap
                    for sensor_df in sensor_dfs.values():
                        if not sensor_df.empty:
                            combined_df = pd.merge(combined_df, sensor_df, on="window_idx", how="outer")

                    combined_df['artwork'] = artwork
                    combined_df['room'] = room
                    combined_df['platform'] = platform

                    processed_docs.append(combined_df)

        if not processed_docs:
            print("Nessun documento valido trovato o processato.")
            return pd.DataFrame()

        final_beacon_columns = sorted(list(all_beacon_columns), key=lambda x: (int(x.split('-')[0]), int(x.split('-')[1])) if '-' in x else (999, 999))

        aligned_docs = []
        for df in processed_docs:
            for col in final_beacon_columns:
                if col not in df.columns:
                    df[col] = default_rssi
            if sensors:
                 for sensor in sensors:
                     for axis in ['x', 'y']:
                         sensor_col_name = f"{sensor}_{axis}_mean"
                         if sensor_col_name not in df.columns:
                             df[sensor_col_name] = np.nan
            aligned_docs.append(df)


        # Combina tutti i dati processati
        final_df = pd.concat(aligned_docs, ignore_index=True)

        # Riempi i NaN globalmente dopo la concatenazione
        final_df.fillna(default_rssi, inplace=True) # Usa default_rssi per i beacon mancanti
        # sensor_cols_to_fill = [col for col in final_df.columns if any(s in col for s in (sensors or []))]
        # final_df[sensor_cols_to_fill] = final_df[sensor_cols_to_fill].fillna(valore_default_sensori)


        # Rimuovi colonne non necessarie
        final_df.drop(columns=['window_idx', 'platform'], inplace=True, errors='ignore')

        # Riordina le colonne
        room_artwork_cols = ['room', 'artwork']
        # Beacon columns sono già ordinate in final_beacon_columns
        sensor_cols_sorted = sorted([col for col in final_df.columns if col not in room_artwork_cols + final_beacon_columns])

        final_ordered_cols = final_beacon_columns + sensor_cols_sorted + room_artwork_cols
        # Assicurati che tutte le colonne esistano prima di riordinare
        final_ordered_cols = [col for col in final_ordered_cols if col in final_df.columns]

        final_df = final_df[final_ordered_cols]

        return final_df
    except Exception as e:
        print(f"Errore durante il processamento dei dati JSON: {e}")
        import traceback
        traceback.print_exc()
        return pd.DataFrame()


# Funzione load_session_data
def load_session_data(file_path, min_rssi_length: int = 100, window_seconds: int = 3, window_step: int = 1, sensors: List[str] = None, default_rssi: float = 110.0):
    print(f"Caricamento e processamento di: {file_path}")
    if not os.path.exists(file_path):
        print(f"Errore: File non trovato - {file_path}")
        return pd.DataFrame()
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)
        if not isinstance(data, list):
             print(f"Attenzione: Il file JSON {file_path} non contiene una lista. Lo incapsulo in una lista.")
             data = [data]

        df = process_json_data(data, min_rssi_length, window_seconds, window_step, sensors, default_rssi)
        return df
    except json.JSONDecodeError as e:
        print(f"Errore nel decodificare JSON da {file_path}: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"Errore inaspettato durante il caricamento di {file_path}: {e}")
        return pd.DataFrame()

def main():
    print("Avvio della creazione dei fingerprint basati sulla MEDIANA...")

    os.makedirs(output_dir, exist_ok=True)

    # Lista dei file raw da processare
    raw_files = [
        "android_session_1_1.json",
        "android_session_2_2.json",
        "ios_session_1_3.json",
        "ios_session_2_4.json"
    ]

    for raw_file in raw_files:
        raw_file_path = os.path.join(raw_data_dir, raw_file)
        print("-" * 30)

        base_name = raw_file.replace('.json', '')
        output_file_name = f"{base_name}_median.csv"
        output_file_path = os.path.join(output_dir, output_file_name)
        df = load_session_data(raw_file_path)

        if not df.empty:
            df.to_csv(output_file_path, index=False)
            print(f"Salvataggio completato: {output_file_path} ({df.shape[0]} righe)")
        else:
            print(f"Nessun dato valido generato per {raw_file}, file CSV non creato.")

    print("-" * 30)
    print("Processamento completato con successo!")

if __name__ == "__main__":
    main()